# Dataset and DataLoader

This notebook shows how MedImageFlow represents image cases with `Sample`, builds a dataset, applies simple transforms, and creates a PyTorch DataLoader.

## Learning objectives

- Describe one image case with `Sample`.
- Build samples lazily from records with `MappingSampleSource`.
- Configure a whole-image `MedicalImageDataset`.
- Apply a field transform to one image array.
- Inspect a single item and a collated PyTorch batch.

## Requirements

Install the notebook dependencies from the repository root:

```bash
python -m pip install -e ".[notebooks]"
```

The dataset itself uses `.npy` files and does not require imaging backends. The DataLoader section uses PyTorch through `create_dataloader`.

## Example data

This tutorial creates synthetic local arrays under `examples/data/generated/dataset/`. No patient data is required.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from medimageflow.data import (
    MappingSampleSource,
    MedicalImageDataset,
    NumpyReader,
    Sample,
    create_dataloader,
)
from medimageflow.transforms import MinMaxNormalize

## 2. Configuration

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "medimageflow").exists():
            return path
    raise RuntimeError("Could not find the MedImageFlow repository root.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "examples" / "data" / "generated" / "dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_DIR)

## 3. Load or create data

The records below mimic rows from a CSV or database. `MappingSampleSource` converts each record to a `Sample` only when that index is requested.

In [ ]:
records = []

for index in range(3):
    rows, cols = np.indices((32, 32))
    image = (rows + cols + index * 8).astype(np.float32)
    label = (((rows - 16) ** 2 + (cols - 16) ** 2) < (6 + index) ** 2).astype(np.uint8)

    image_path = DATA_DIR / f"case-{index:03d}-image.npy"
    label_path = DATA_DIR / f"case-{index:03d}-label.npy"
    np.save(image_path, image)
    np.save(label_path, label)

    records.append(
        {
            "case_id": f"case-{index:03d}",
            "image_path": image_path,
            "label_path": label_path,
            "age": 50 + index,
            "site": "synthetic",
        }
    )

records[0]

## 4. Run the MedImageFlow workflow

`MedicalImageDataset` reads full images. `NumpyReader` is built in, but it is shown explicitly here to demonstrate reader configuration.

In [ ]:
source = MappingSampleSource(
    records,
    paths={"image": "image_path", "label": "label_path"},
    features=("age",),
    id="case_id",
    metadata=("site",),
)

dataset = MedicalImageDataset(
    source,
    readers=[NumpyReader()],
    image_field_transforms={"image": MinMaxNormalize()},
)

len(dataset)

A direct `Sample` is also valid when you already have paths in Python.

In [ ]:
single_sample = Sample(
    paths={"image": records[0]["image_path"], "label": records[0]["label_path"]},
    features={"age": records[0]["age"]},
    id=records[0]["case_id"],
)
single_sample

## 5. Inspect results

A dataset item is a dictionary with `id`, `metadata`, `features`, and the loaded image fields.

In [ ]:
item = dataset[0]

print(item.keys())
print("id:", item["id"])
print("image:", item["image"].shape, item["image"].dtype, item["image"].min(), item["image"].max())
print("label:", item["label"].shape, item["label"].dtype)
print("features:", item["features"])
print("metadata:", item["metadata"])

## 6. Create a DataLoader

`create_dataloader` wraps `torch.utils.data.DataLoader`. Numeric NumPy arrays are collated into tensors by PyTorch's default collate function.

In [ ]:
try:
    loader = create_dataloader(dataset, batch_size=2, shuffle=False)
    batch = next(iter(loader))
except ImportError as error:
    loader = None
    batch = None
    print(error)

if batch is not None:
    print(batch.keys())
    print("ids:", batch["id"])
    print("image:", batch["image"].shape, batch["image"].dtype)
    print("label:", batch["label"].shape, batch["label"].dtype)
    print("age:", batch["features"]["age"].shape, batch["features"]["age"].dtype)

## 7. Visualize results

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(item["image"], cmap="gray")
axes[0].set_title("image")
axes[1].imshow(item["label"], cmap="gray")
axes[1].set_title("label")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
figure

## 8. Next steps

Use `MedicalImageDataset.from_csv(...)` when your sample table is stored as CSV. Use `PatchDataset` when you need model-ready crops instead of complete images.